# Study 2: Lactate Response Figure
**IPC vs. Sham — Blood Lactate Across Timepoints**

This notebook generates the lactate response figure (2-panel: Baseline vs. Trial) and a descriptive statistics table for Study 2.

**Data required:** place `Spreadsheet_Study_2.xlsx` inside a `data/` folder in the working directory.

**Note:** This notebook was originally developed in Google Colab. Colab-specific calls (`drive.mount`, `files.download`) have been removed; figures are saved locally instead.

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Use Arial / closest available sans-serif
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans', 'sans-serif']

## Load and Clean Data

In [ ]:
# Place data file at: data/Spreadsheet_Study_2.xlsx
df = pd.read_excel("data/Spreadsheet_Study_2.xlsx")

# Remove non-study participant
df = df[df['Subjects'].astype(str).str.strip() != 'Brian'].copy()

## Prepare Plotting Data

In [ ]:
# Column name maps: timepoint label → spreadsheet column
t1_map = {
    'T1_Rest_Lac':    'Rest',
    'T1_WarmUp_Lac':  'Pre-3600m',
    'T1Post3600Lac':  'Post-3600m',
    'T1Pre1200Lac':   'Pre-1200m',
    'T1Post1200Lac':  'Post-1200m'
}

t2_map = {
    'T2RestLac':      'Rest',
    'T2WarmUpLac':    'Pre-3600m',
    'T2Post3600Lac':  'Post-3600m',
    'T2Pre1200Lac':   'Pre-1200m',
    'T2Post1200Lac':  'Post-1200m'
}

all_cols = list(t1_map.keys()) + list(t2_map.keys())
for col in all_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Split by trial group
ipc_data  = df[df['trial'] == 'IPC']
sham_data = df[df['trial'] == 'Sham']

def get_stats(data, col_map):
    valid_cols = [c for c in col_map if c in data.columns]
    if not valid_cols:
        return None
    return {
        'mean': data[valid_cols].mean().values,
        'sd':   data[valid_cols].std().values,
        'time': [col_map[c] for c in valid_cols]
    }

ipc_t1  = get_stats(ipc_data,  t1_map)
ipc_t2  = get_stats(ipc_data,  t2_map)
sham_t1 = get_stats(sham_data, t1_map)
sham_t2 = get_stats(sham_data, t2_map)

## Figure: Lactate Response (Baseline vs. Trial)
Panel A = Baseline (T1), Panel B = Trial (T2). Lines show mean ± SD.

In [ ]:
TITLE_SIZE  = 22
LABEL_SIZE  = 18
TICK_SIZE   = 16
LEGEND_SIZE = 16
LINE_WIDTH  = 2.5
MARKER_SIZE = 10

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)

def plot_panel(ax, ipc_stats, sham_stats, title):
    if ipc_stats:
        ax.errorbar(x=ipc_stats['time'], y=ipc_stats['mean'], yerr=ipc_stats['sd'],
                    label='IPC', fmt='-s', color='gray',
                    capsize=6, markerfacecolor='gray',
                    markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
    if sham_stats:
        ax.errorbar(x=sham_stats['time'], y=sham_stats['mean'], yerr=sham_stats['sd'],
                    label='Sham', fmt='-o', color='black',
                    capsize=6, markerfacecolor='white',
                    markeredgewidth=2.5, markersize=MARKER_SIZE, linewidth=LINE_WIDTH)
    ax.set_title(title, fontsize=TITLE_SIZE, fontweight='bold')
    ax.set_xlabel("Timepoint", fontsize=LABEL_SIZE, fontweight='bold')
    ax.tick_params(axis='both', which='major', labelsize=TICK_SIZE)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(frameon=False, fontsize=LEGEND_SIZE)
    ax.grid(False)

plot_panel(axes[0], ipc_t1, sham_t1, "A. Baseline (T1)")
axes[0].set_ylabel("Lactate (mmol/L)", fontsize=LABEL_SIZE, fontweight='bold')

plot_panel(axes[1], ipc_t2, sham_t2, "B. Trial (T2)")

sns.despine()
plt.tight_layout()
plt.savefig('Lactate_Response_Baseline_vs_Trial.pdf', dpi=300, bbox_inches='tight')
plt.savefig('Lactate_Response_Baseline_vs_Trial.png', dpi=300, bbox_inches='tight')
plt.show()
print("Figure saved as PDF and PNG.")

## Descriptive Statistics Table

In [ ]:
variable_map = {
    "Resting lactate":   ["T1_Rest_Lac",   "T2RestLac"],
    "Pre-3600m lactate": ["T1_WarmUp_Lac", "T2WarmUpLac"],
    "Post-3600m lactate":["T1Post3600Lac", "T2Post3600Lac"],
    "Pre-1200m lactate": ["T1Pre1200Lac",  "T2Pre1200Lac"],
    "Post-1200m lactate":["T1Post1200Lac", "T2Post1200Lac"],
    "3600m time (s)":    ["baseline3600msec", "36000msec2"],
    "1200m time (s)":    ["baseline1200msec", "1200msec2"],
    "Critical speed":    ["precriticalspeed", "postcriticalspeed"],
    "D prime":           ["Pred",             "postd"]
}

results = []
for trial_name, group_df in df.groupby('trial'):
    for var_name, (t1_col, t2_col) in variable_map.items():
        for col_label, col in [("Baseline", t1_col), ("Trial", t2_col)]:
            if col in group_df.columns:
                vals = pd.to_numeric(group_df[col], errors='coerce')
                results.append({
                    "Group":    trial_name,
                    "Variable": var_name,
                    "Visit":    col_label,
                    "Mean":     round(vals.mean(), 2),
                    "SD":       round(vals.std(), 2)
                })

stats_df = pd.DataFrame(results)
for group_name in stats_df['Group'].unique():
    print(f"\n--- {group_name} ---")
    print(stats_df[stats_df['Group'] == group_name]
          [['Variable', 'Visit', 'Mean', 'SD']]
          .to_string(index=False))